[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# Proporciones, variables categóricas y más de dos grupos.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se probarán proporciones y la independencia de dos variables categóricas con la prueba *z* y la chi cuadrada, y se compararán más de dos grupos con ANOVA, Levene, Welch y Tukey.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [2]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

((342, 7), (891, 15), (244, 8))

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda definirá `d_cohen`, presentada en `20_pruebas_para_dos_grupos.ipynb`.

In [3]:
def d_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    varianza_combinada = ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(varianza_combinada)

---
## 1. Proporciones y variables categóricas (20 min).

* **Prueba *z* de dos proporciones** (`proportions_ztest(conteos, totales)`): ¿difiere la proporción de éxito entre dos grupos?
* **Chi cuadrada de independencia** (`stats.chi2_contingency(tabla)`): ¿están asociadas dos variables categóricas? Compara la tabla observada con la **esperada** si fueran independientes.
* **Supuesto clave:** frecuencias **esperadas** suficientemente grandes (en general ≥ 5 en cada celda). Si no se cumple, se agrupan categorías o se usa la prueba exacta de Fisher (`stats.fisher_exact`, para tablas 2×2).

**Concepto: prueba *z* de dos proporciones.** Si H0 fuera cierta (los dos grupos tienen la misma proporción), se juntan los dos grupos para estimar esa proporción común. Luego se mide la diferencia entre las dos proporciones observadas **en errores estándar**, igual que la *t* con las medias. Por el TCL, ese número se compara con la campana normal: más allá de unos 2 errores estándar, la diferencia es difícil de atribuir al azar.

**Ejemplo:** si 30% de los clientes de una sucursal y 35% de otra recomiendan el servicio, la prueba responde si 5 puntos de diferencia son mucho o poco **para el número de clientes encuestados**.

* La siguiente celda comparará con la prueba *z* la tasa de supervivencia de mujeres y hombres en el Titanic.

In [4]:
tabla_sexo = pd.crosstab(titanic["sex"], titanic["survived"])
sobrevivientes = tabla_sexo[1].to_numpy()
totales = tabla_sexo.sum(axis=1).to_numpy()
estadistico_z, valor_p_z = proportions_ztest(sobrevivientes, totales)
(sobrevivientes / totales).round(3), round(float(estadistico_z), 2), float(valor_p_z)

(array([0.742, 0.189]), 16.22, 3.7117477701134797e-59)

La diferencia es enorme (74.2% contra 18.9%) y su valor *p* es prácticamente 0. Para reportar la magnitud se usa la **diferencia de proporciones** con su IC (sesión 5) o el **riesgo relativo**: las mujeres sobrevivieron casi 4 veces más que los hombres.

**Ejemplo:** ¿están asociadas la clase y la supervivencia? Se revisa la tabla esperada antes de interpretar.

**Concepto: chi cuadrada.** Primero se construye la tabla **esperada**: cuántos casos habría en cada celda si las dos variables no tuvieran relación. Para cada celda, se multiplica el total de su fila por el total de su columna y se divide entre el total general. Luego la chi cuadrada suma, celda por celda, qué tan lejos está lo observado de lo esperado (en relación con lo esperado). Si las variables no tienen relación, la suma queda chica.

**Ejemplo:** si 38% de todos los pasajeros sobrevivió y había 216 en primera clase, sin relación entre clase y supervivencia se esperarían unos 83 sobrevivientes de primera (216 × 0.384). Si se observan muchos más, eso aporta mucho a la chi cuadrada.

Su tamaño del efecto es la **V de Cramér**, que va de 0 (sin relación) a 1 (relación perfecta).

* La siguiente celda aplicará la chi cuadrada de independencia a `class` contra `survived` y mostrará la tabla esperada.

In [5]:
tabla_clase = pd.crosstab(titanic["class"], titanic["survived"])
chi2 = stats.chi2_contingency(tabla_clase)
esperada = pd.DataFrame(chi2.expected_freq, index=tabla_clase.index, columns=tabla_clase.columns).round(1)
round(float(chi2.statistic), 1), int(chi2.dof), float(chi2.pvalue), esperada

(102.9,
 2,
 4.549251711298793e-23,
 survived      0      1
 class                 
 First     133.1   82.9
 Second    113.4   70.6
 Third     302.5  188.5)

Todas las frecuencias esperadas superan 5, así que la aproximación es válida. La chi cuadrada dice que hay asociación, pero no cuánta: el tamaño del efecto es la **V de Cramér** (0 = independencia, 1 = asociación perfecta), que se calcula con `stats.contingency.association(tabla, method="cramer")`.

| V de Cramér con 1 grado de libertad mínimo (orientativo) | 0.1 | 0.3 | 0.5 |
|---|---|---|---|
| Magnitud | Pequeña | Mediana | Grande |

📖 **Para profundizar (opcional).** *Las fórmulas de la sección 1.*
* Prueba *z* de dos proporciones: bajo H0 ($p_A = p_B$) se estima la proporción común $\hat{p}$ con los dos grupos juntos y $$z = \frac{\hat{p}_A - \hat{p}_B}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}$$
* Chi cuadrada: la frecuencia esperada de cada celda es $E_{ij} = \dfrac{(\text{total de la fila } i)(\text{total de la columna } j)}{n}$ y $$\chi^2 = \sum_{i,j} \frac{(O_{ij} - E_{ij})^2}{E_{ij}}, \qquad g.l. = (f - 1)(c - 1)$$
* V de Cramér: $V = \sqrt{\dfrac{\chi^2}{n\,(\min(f, c) - 1)}}$.

**💡 Ideas clave de la sección 1:**
* Prueba *z* para dos proporciones; chi cuadrada para dos categóricas, revisando que las frecuencias esperadas sean ≥ 5.
* La magnitud se reporta con la diferencia de proporciones, el riesgo relativo o la V de Cramér.

### 🟢 Ejercicio 1 · Mujeres y hombres en tercera clase.

**🎯 Objetivo:** aplicar la prueba *z* y la V de Cramér dentro de un subgrupo. **Qué se espera aprender:** a revisar si una asociación global se mantiene al controlar por otra variable.

¿La ventaja de las mujeres se mantiene en tercera clase? Con los pasajeros de `class == "Third"`, compara la supervivencia de mujeres y hombres con la prueba *z* y calcula la V de Cramér de la tabla 2×2.

* La siguiente celda muestra una solución posible.

In [6]:
tercera = titanic[titanic["class"] == "Third"]
tabla_tercera = pd.crosstab(tercera["sex"], tercera["survived"])
z, valor_p = proportions_ztest(tabla_tercera[1].to_numpy(), tabla_tercera.sum(axis=1).to_numpy())
(tabla_tercera, (tabla_tercera[1] / tabla_tercera.sum(axis=1)).round(3), round(float(z), 2), float(valor_p),
 round(float(stats.contingency.association(tabla_tercera, method="cramer")), 3))

(survived    0   1
 sex              
 female     72  72
 male      300  47,
 sex
 female    0.500
 male      0.135
 dtype: float64,
 8.58,
 9.300850234684943e-18,
 0.387)

**✅ Respuesta sugerida:**

En tercera clase sobrevivió 50.0% de las mujeres y 13.5% de los hombres (*z* = 8.58, *p* < 0.001; V de Cramér = 0.39, una asociación mediana a grande). La ventaja de las mujeres se mantiene dentro de la clase, aunque la tasa de las mujeres de tercera es mucho menor que la de primera (97%).

---
## 2. Más de dos grupos (25 min).

Comparar tres grupos con tres pruebas *t* infla el error tipo I (sección 2 de `19_logica_del_contraste.ipynb`). El procedimiento es:

1. **ANOVA de una vía** (`stats.f_oneway`): H0 = todas las medias son iguales. Supone independencia, normalidad aproximada y varianzas iguales.
2. **Levene** (`stats.levene`): ¿son iguales las varianzas? Si no, se usa el **ANOVA de Welch** (`anova_oneway(..., use_var="unequal")` de *statsmodels*).
3. Si se rechaza H0, **post hoc de Tukey** (`pairwise_tukeyhsd`) para saber **qué** pares difieren, controlando el error de las comparaciones múltiples.
4. Tamaño del efecto: **η²** = suma de cuadrados entre grupos / suma de cuadrados total (proporción de la variabilidad explicada por el grupo).

**Concepto: ANOVA y η².** La variabilidad total de los datos se separa en dos partes: la que hay **entre** grupos (qué tan distintas son las medias de los grupos) y la que hay **dentro** de los grupos (qué tan distintos son los individuos de un mismo grupo).

El estadístico *F* compara ambas partes: si los grupos no difieren, las dos son parecidas y *F* queda cerca de 1; si las medias de los grupos están muy separadas en comparación con la variación dentro de cada uno, *F* crece. El tamaño del efecto, **η²**, es la fracción de la variabilidad total que se explica por el grupo: un η² de 0.7 quiere decir que 70% de las diferencias entre individuos se debe a a qué grupo pertenecen.

**Ejemplo:** en la masa de los pingüinos, la especie explica alrededor de 67% de las diferencias entre individuos (η² ≈ 0.67), porque los Gentoo pesan mucho más. En cambio, en el ejemplo de esta sección el día de la semana casi no explica la propina porcentual.

* La siguiente celda definirá `eta_cuadrada` a partir de los valores y las etiquetas de grupo.

In [7]:
def eta_cuadrada(valores, grupos):
    datos = pd.DataFrame({"valor": valores, "grupo": grupos}).dropna()
    media_global = datos["valor"].mean()
    por_grupo = datos.groupby("grupo")["valor"].agg(["count", "mean"])
    suma_entre = (por_grupo["count"] * (por_grupo["mean"] - media_global) ** 2).sum()
    suma_total = ((datos["valor"] - media_global) ** 2).sum()
    return suma_entre / suma_total

**Ejemplo:** se usará `tip_pct` por día de la semana en `tips`, un caso donde **no** se espera una diferencia grande.

* La siguiente celda aplicará Levene, el ANOVA clásico, el ANOVA de Welch y Kruskal-Wallis (alternativa por rangos) a `tip_pct` por día, y calculará η².

In [8]:
grupos_dia = [grupo["tip_pct"].to_numpy() for _, grupo in propinas.groupby("day")]
welch_dia = anova_oneway(propinas["tip_pct"], propinas["day"], use_var="unequal")
pd.DataFrame({
    "valor p": [stats.levene(*grupos_dia).pvalue, stats.f_oneway(*grupos_dia).pvalue,
                welch_dia.pvalue, stats.kruskal(*grupos_dia).pvalue],
}, index=["Levene", "ANOVA", "ANOVA de Welch", "Kruskal-Wallis"]).round(3), round(float(eta_cuadrada(propinas["tip_pct"], propinas["day"])), 3)

(                valor p
 Levene            0.170
 ANOVA             0.469
 ANOVA de Welch    0.415
 Kruskal-Wallis    0.607,
 0.01)

Ninguna prueba encuentra diferencias y η² es casi 0: el día explica apenas alrededor de 1% de la variabilidad de la propina porcentual. Como el ANOVA no rechaza, **no** se hace Tukey: buscar pares significativos después de un ANOVA no significativo es otra forma de *p-hacking*.

| η² (orientativo) | 0.01 | 0.06 | 0.14 |
|---|---|---|---|
| Magnitud | Pequeño | Mediano | Grande |

**Consecuencia: significativo no es lo mismo que importante.** Con muestras enormes, cualquier diferencia mínima resulta significativa. Se simularán dos grupos de 100,000 personas cuyas medias difieren 0.45 puntos en una escala con desviación estándar de 15 (como la de muchas pruebas de inteligencia).

* La siguiente celda calculará el valor *p* de Welch y la d de Cohen de esa diferencia mínima.

In [9]:
enorme_a = rng.normal(100, 15, size=100_000)
enorme_b = rng.normal(100.45, 15, size=100_000)
float(stats.ttest_ind(enorme_b, enorme_a, equal_var=False).pvalue), round(float(d_cohen(enorme_b, enorme_a)), 3)

(7.973271311876997e-06, 0.02)

El valor *p* es diminuto, pero la d es ≈ 0.02 (la diferencia verdadera es 0.45 / 15 = 0.03 desviaciones estándar): un efecto **despreciable**. Reportar solo "*p* < 0.001" llevaría a creer que es un hallazgo importante. Por eso el tamaño del efecto y su IC son obligatorios en el informe.

📖 **Para profundizar (opcional).** *Las fórmulas del ANOVA (sección 2).* La suma de cuadrados total se descompone así:

$$SC_{\text{total}} = \sum_{i}(x_i - \bar{x})^2 = \underbrace{\sum_{g} n_g(\bar{x}_g - \bar{x})^2}_{SC_{\text{entre}}} + \underbrace{\sum_{g}\sum_{i \in g}(x_i - \bar{x}_g)^2}_{SC_{\text{dentro}}}$$

Con $k$ grupos y $n$ datos, $F = \dfrac{SC_{\text{entre}} / (k - 1)}{SC_{\text{dentro}} / (n - k)}$ y $\eta^2 = SC_{\text{entre}} / SC_{\text{total}}$.

**💡 Ideas clave de la sección 2:**
* ANOVA (Welch si Levene rechaza) → Tukey solo si el ANOVA rechaza → η².
* Kruskal-Wallis es la alternativa por rangos.
* Con *n* enorme todo es significativo: el tamaño del efecto decide si importa.

### 🔴 Reto 1 · Doce pruebas a la vez.

**🎯 Objetivo:** aplicar correcciones por comparaciones múltiples a un caso real. **Qué se espera aprender:** cuándo las correcciones cambian las conclusiones y cuándo no.

Para cada especie y cada una de las cuatro medidas, compara machos y hembras con Welch (12 pruebas). Aplica las correcciones de Bonferroni y de Holm con `multipletests`. ¿Cuántas pruebas siguen siendo significativas? Agrega la d de Cohen de cada comparación: ¿cuál es la diferencia entre sexos más pequeña?

* La siguiente celda muestra una solución posible.

In [10]:
filas = []
for especie, datos in pinguinos.groupby("species"):
    for medida in medidas:
        machos = datos.loc[datos["sex"] == "Male", medida].dropna()
        hembras = datos.loc[datos["sex"] == "Female", medida].dropna()
        filas.append({"especie": especie, "medida": medida,
                      "p": stats.ttest_ind(machos, hembras, equal_var=False).pvalue,
                      "d": d_cohen(machos, hembras)})
doce = pd.DataFrame(filas)
doce["rechaza_bonferroni"] = multipletests(doce["p"], method="bonferroni")[0]
doce["rechaza_holm"] = multipletests(doce["p"], method="holm")[0]
doce.sort_values("d").round({"d": 2})

,especie,medida,p,d,rechaza_bonferroni,rechaza_holm
2,Adelie,flipper_length_mm,1.109975e-05,0.75,True,True
7,Chinstrap,body_mass_g,2.264432e-06,1.26,True,True
6,Chinstrap,flipper_length_mm,2.535148e-07,1.39,True,True
0,Adelie,bill_length_mm,4.801082e-15,1.45,True,True
1,Adelie,bill_depth_mm,1.914093e-15,1.48,True,True
8,Gentoo,bill_length_mm,1.315039e-14,1.62,True,True
10,Gentoo,flipper_length_mm,7.112382e-17,1.81,True,True
4,Chinstrap,bill_length_mm,8.918409e-10,1.84,True,True
5,Chinstrap,bill_depth_mm,6.600016e-13,2.16,True,True
3,Adelie,body_mass_g,6.402320e-26,2.17,True,True


**✅ Respuesta sugerida:**

Las 12 comparaciones son significativas incluso con Bonferroni y con Holm (el mayor *p* es ≈ 1 × 10⁻⁵, muy por debajo de 0.05/12 ≈ 0.004). La diferencia más pequeña es la aleta de los Adelie (d = 0.75, un efecto mediano a grande); las mayores son la masa de los Gentoo (d = 2.70) y la profundidad del pico de los Gentoo (d = 2.27). Con efectos tan grandes la corrección no cambia nada; importa cuando hay muchos *p* cercanos a 0.05.

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios 🟢 pendientes de este *notebook* y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `22_tamano_del_efecto.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>